In [28]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

BASE_DIR = Path.cwd().parent
DB_PATH  = BASE_DIR / 'database' / 'olist.db'

engine = create_engine(f'sqlite:///{DB_PATH}')

In [29]:
# Understand structure of tables by looking at first few row of each table  
# Help us understand what data we have and how to work with other data
# LIMIT 2 = only return 2 rows, just enough to see column names and sample values

tables = [
    'customers', 'orders', 'order_items',
    'order_payments', 'order_reviews', 'products', 'sellers'
]

for x in tables:
    print(f"\n{'='*55}")
    print(f" TABLE: {x}")
    print('='*55)
    table = pd.read_sql(f"SELECT * FROM {x} LIMIT 2", engine)
    print(table.to_string())


 TABLE: customers
                        customer_id                customer_unique_id customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                    14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                     9790  sao bernardo do campo             SP

 TABLE: orders
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27  

In [30]:

# SUBSTR(column,start,Length)= cut as month for us to group by month,every month instead of every timesstamp
# GROUP BY month = grouping
# ORDER BY = sorting or accending month,ealier month first
query = """

SELECT
    SUBSTR(o.order_purchase_timestamp,1,7)  AS month,
    COUNT(DISTINCT o.order_id)                 AS total_orders,
    ROUND(SUM(p.payment_value), 2)             AS total_revenue,
    ROUND(AVG(p.payment_value), 2)             AS avg_order_value

FROM orders o           -- orders table is now nicknamed 'o'
JOIN order_payments p   -- order_payments table is now nicknamed 'p'
    ON o.order_id = p.order_id

WHERE o.order_status = 'delivered'

GROUP BY month 
ORDER BY month
"""

monthly_revenue = pd.read_sql(query, engine)
print("="*80)
print("Monthly Revenue Trends")
print("="*80)
# Save the query to sql file for later use by using w(write) instead of r(read) to open scheama create table,w to wirte save the query
with open(BASE_DIR / 'database' / 'queries' / 'monthly_revenue.sql', 'w') as f:
    f.write(query)
monthly_revenue

Monthly Revenue Trends


,month,total_orders,total_revenue,avg_order_value
0,2016-10,265,46566.71,165.13
1,2016-12,1,19.62,19.62
2,2017-01,750,127545.67,159.63
3,2017-02,1653,271298.65,155.12
4,2017-03,2546,414369.39,153.47
5,2017-04,2303,390952.18,160.49
6,2017-05,3546,567066.73,149.74
7,2017-06,3135,490225.60,147.53
8,2017-07,3872,566403.93,136.58
9,2017-08,4193,646000.61,147.05


In [31]:
# use julianday() convert date to numeric value, then we can do math to calculate how many days to deliver how many days early or late 

query = """
SELECT
    order_status,
    COUNT(order_id)    AS order_count,
    ROUND(AVG(
        julianday(order_delivered_customer_date) -
        julianday(order_purchase_timestamp)
    ), 1)              AS avg_days_to_deliver,
    ROUND(AVG(
        julianday(order_estimated_delivery_date) -
        julianday(order_delivered_customer_date)
    ), 1)              AS avg_days_early_or_late
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL
GROUP BY order_status
ORDER BY order_count DESC
"""
delivery_performance = pd.read_sql(query,engine)
# Top 10 product categories by revenue
print("="*80)
print("Delivery Performance by Order Status")
print("\n-Late show a negative value,and early show a positive value")
print("="*80)

with open(BASE_DIR / 'database' / 'queries' / 'delivery_performance.sql', 'w') as f:
    f.write(query)
delivery_performance




Delivery Performance by Order Status

-Late show a negative value,and early show a positive value


,order_status,order_count,avg_days_to_deliver,avg_days_early_or_late
0,delivered,96470,12.6,11.2
1,canceled,6,20.4,27.2


In [32]:
# Top 10 product categories by revenue
query = """
SELECT
    pr.product_category_name           AS category,
    COUNT(DISTINCT oi.order_id)         AS total_orders,
    ROUND(SUM(oi.price), 2)             AS total_revenue,
    ROUND(AVG(oi.price), 2)             AS avg_item_price,
    ROUND(AVG(oi.freight_value), 2)     AS avg_freight
FROM order_items oi
JOIN products pr ON oi.product_id = pr.product_id
WHERE pr.product_category_name IS NOT NULL
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10
"""
top10_categories = pd.read_sql(query,engine)
print("="*80)
print("Top 10 Product Categories by Revenue")
print("="*80)

with open(BASE_DIR / 'database' / 'queries' / 'top_categories.sql', 'w') as f:
    f.write(query)

top10_categories

Top 10 Product Categories by Revenue


,category,total_orders,total_revenue,avg_item_price,avg_freight
0,beleza_saude,8836,1258681.34,130.16,18.88
1,relogios_presentes,5624,1205005.68,201.14,16.78
2,cama_mesa_banho,9417,1036988.68,93.30,18.42
3,esporte_lazer,7720,988048.97,114.34,19.51
4,informatica_acessorios,6689,911954.32,116.51,18.82
5,moveis_decoracao,6449,729762.49,87.56,20.73
6,cool_stuff,3632,635290.85,167.36,22.14
7,utilidades_domesticas,5884,632248.66,90.79,20.99
8,automotivo,3897,592720.11,139.96,21.88
9,ferramentas_jardim,3518,485256.46,111.63,22.77


In [33]:
query = """
SELECT
    payment_type,
    COUNT(*)                            AS transaction_count,
    ROUND(SUM(payment_value), 2)        AS total_value,
    ROUND(AVG(payment_value), 2)        AS avg_payment,
    ROUND(AVG(payment_installments), 1) AS avg_installments
FROM order_payments
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY transaction_count DESC
"""
print("="*80)
print("Payment Method Analysis")
print("="*80)
payment_analysis = pd.read_sql(query,engine)

with open(BASE_DIR / 'database' / 'queries' / 'payment_methods.sql', 'w') as f:
    f.write(query)

payment_analysis


Payment Method Analysis


,payment_type,transaction_count,total_value,avg_payment,avg_installments
0,credit_card,76795,12542084.19,163.32,3.5
1,boleto,19784,2869361.27,145.03,1.0
2,voucher,5775,379436.87,65.70,1.0
3,debit_card,1529,217989.79,142.57,1.0


In [34]:
query = """
SELECT
    c.customer_state                                              AS state,
    COUNT(DISTINCT o.order_id)                                    AS total_orders,
    ROUND(SUM(p.payment_value), 2)                                AS total_revenue,
    ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2)  AS revenue_per_order,
    ROUND(AVG(r.review_score), 2)                                 AS avg_review_score
FROM orders o
JOIN customers c          ON o.customer_id = c.customer_id
JOIN order_payments p     ON o.order_id    = p.order_id
LEFT JOIN order_reviews r ON o.order_id    = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 15
"""
state_performance = pd.read_sql(query,engine)
print("="*80)
print("State Performance by Revenue")
print("="*80)
state_performance

with open(BASE_DIR / 'database' / 'queries' / 'state_performance.sql', 'w') as f:
    f.write(query)

state_performance

State Performance by Revenue


,state,total_orders,total_revenue,revenue_per_order,avg_review_score
0,SP,40500,5786415.41,142.87,4.25
1,RJ,12350,2059006.91,166.72,3.97
2,MG,11354,1822216.31,160.49,4.19
3,RS,5345,863452.68,161.54,4.18
4,PR,4923,783475.26,159.15,4.24
5,SC,3546,596061.93,168.09,4.13
6,BA,3256,591917.44,181.79,3.93
7,DF,2080,347185.42,166.92,4.13
8,GO,1957,335831.81,171.61,4.09
9,ES,1995,318210.80,159.50,4.08
